Running example: Bivalirudin (DrugBank DB00006, ChEMBL parent CHEMBL2103749, a direct thrombin inhibitor - the textbook narrow-profile drug. 

We will build: 
1. **T_mech** from the ChEMBL mechanism table, represents curated mechanism-of-action targets
2. **T_bio** ChEMBL activity table, represents bioactivity targets with pChEMBL >= 6

Both returned as set of UniProt accessions, ready to union with DrugBank-derived T_BD. 
Requires internet access to the ChEMBL API. 

In [41]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated.*")
import requests
import pandas as pd
from chembl_webresource_client.new_client import new_client

In [42]:
molecule  = new_client.molecule
mechanism = new_client.mechanism
activity  = new_client.activity
target    = new_client.target

pd.set_option("display.max_colwidth", 60)
print("ChEMBL client ready")

ChEMBL client ready


### Step 1 - Resolve the drug to ChEMBL molecule record(s)
* Use Drug name instead of DrugBank ID. 
* Look at all matching molecule record, to expose the parent/child salt-form issue
* Always resolve to the parent

In [43]:
DRUG = "bivalirudin" # [BD00006, bivalirudin], [DB00014, Goserelin], use <drug>/<name>
THETA = 6.0 # pChEMBL cutoff for the bioactivity layer (6= 1 μM)

hits = list(
    molecule.filter(molecule_synonyms__molecule_synonym__iexact=DRUG).only(["molecule_chembl_id", "pref_name"])
)

PARENT = hits[0]["molecule_chembl_id"]
PREF = hits[0]["pref_name"]

print(f"{DRUG}  ->  {PARENT}  ({PREF})")

if not hits:
    raise ValueError(f"No ChEMBL match for {DRUG!r}")
PARENT = hits[0]["molecule_chembl_id"]

bivalirudin  ->  CHEMBL5314348  (BIVALIRUDIN)


### Two tables that link a molecule to targets

#### 1. drug_mechanism table - curated, low volume
* This is human-curated mechanism-of-action: "this drug works by doing X to target Y."
* Each row carries [target_chembl_id, action_type, mechanism_of_action]
* Efficacy targets
* Feed T_mech

    **Info to keep**
  * target_chembl_id - the link to UniPort via the target table.
  * action_type - Agonist/Inhibitor/etc. Useful both as an edge attribute and later if there is a need to distinguish direction of action
  * mechanism_of_action - human-readable text, keep for the audit trail.
  * direct_interaction (1/0) - whether the drug binds the target directly.
  * molecular_mechanism (1/0) - describes the molecular target (==1)  rather than a higher-level physiological mechanism (==0)

#### 2. bioactivity table - measured, high volume
* Experimental assay data: "in some experiments, this compound was tested against target Y and the measured potency was Z."
* Each row is one measurement [taregt_chembl_id, pchembl_value, standard_type]
* A well-studied drug can have hundreds of these rows across many targets, including off-targets and counter-screens.
* "What has this been tested against and how strongly did it bind."
* Feed T_bio

    **Info to keep**
  * target_chembl_id - resolves to UniProt downstream.
  * target_organism - filter to human; drop everything else.
  * pchembl_value - cutoff variable
  * standard_type - Ki/IC50/Kd/EC50; tells what the potency means
  * standard_relation - the = vs >/< qualifier. A high standard_value with > is the opposite of a hit.
  * assay_type - Filter to binding (and maybe functional).
  * standard_flag - The value was successfully standardized/validated.
  * data_validity_comment - non-null means ChEMBL flagged a problem.
  * potential_duplicate - flags likely  duplicate measurements.



In [45]:
mech_rows = list(
    mechanism.filter(parent_molecule_chembl_id=PARENT)
                .only(["target_chembl_id", "action_type", "mechanism_of_action",
                    "direct_interaction", "molecular_mechanism"])
) 

mech_rows = [m for m in mech_rows
             if m.get("target_chembl_id")
             and m.get("molecular_mechanism")
             and m.get("direct_interaction")]

for m in mech_rows:
    print(m["target_chembl_id"], m["action_type"], "|", m["mechanism_of_action"])
print(f"{len(mech_rows)} row(s) after filtering")

CHEMBL204 INHIBITOR | Thrombin inhibitor
1 row(s) after filtering


In [47]:
KEEP = ["target_chembl_id", "target_organism",
        "pchembl_value", "standard_type", "standard_relation",
        "assay_type", "standard_flag", "data_validity_comment",
        "potential_duplicate", "assay_chembl_id"]

act_full = list(activity.filter(parent_molecule_chembl_id=PARENT))
act_df = pd.DataFrame(act_full)
act_df = act_df[[c for c in KEEP if c in act_df.columns]]   # keep only your columns
print(f"activity: {len(act_df)} total row(s)")
act_df

activity: 40 total row(s)


,target_chembl_id,target_organism,pchembl_value,standard_type,standard_relation,assay_type,standard_flag,data_validity_comment,potential_duplicate,assay_chembl_id
0,CHEMBL6020,Homo sapiens,None,IC50,>,A,1,Outside typical range,0,CHEMBL4028921
1,CHEMBL5748,Homo sapiens,None,IC50,>,A,1,Outside typical range,0,CHEMBL4028922
2,CHEMBL5918,Homo sapiens,None,IC50,>,A,1,Outside typical range,0,CHEMBL4028923
3,CHEMBL1743128,Homo sapiens,None,IC50,>,A,1,Outside typical range,0,CHEMBL4028924
4,CHEMBL372,Homo sapiens,None,DILI_severity_class,=,T,0,None,0,CHEMBL4029349
5,CHEMBL372,Homo sapiens,None,DILI_Concern,None,T,0,None,0,CHEMBL4029350
6,CHEMBL4303835,Severe acute respiratory syndrome coronavirus 2,None,Inhibition,=,F,1,None,0,CHEMBL4303805
7,CHEMBL4523582,Severe acute respiratory syndrome coronavirus 2,None,Inhibition,=,F,1,None,0,CHEMBL4495582
8,CHEMBL4303835,Severe acute respiratory syndrome coronavirus 2,None,Inhibition,=,F,1,None,0,CHEMBL4513082
9,CHEMBL4303835,Severe acute respiratory syndrome coronavirus 2,None,Inhibition,=,F,1,None,0,CHEMBL4513082


### Targets to UniProt
The shared target → UniProt hop that both T_mech and T_bio call 
the only thing that differs between them is the single_protein_only flag they pass in.

#### single_protein_only filter
**T_bio** 
* built from raw bioactivity assays
* restricting to SINGLE PROTEIN is a quality filter:
  
**T_mech**  
* It's hand-curated — a person reviewed the literature and asserted "this drug works by acting on this target."
* When a curator records the target as a protein complex or family, that's not noise to be filtered out; that is the biology, recorded deliberately.
* So you want to keep it and pull all the component proteins.

In [39]:
def targets_to_uniprot(target_ids, human_only=True, single_protein_only=False):

    ids = sorted({t for t in target_ids if t})

    out, chunk = {}, 50
    # out   : the result dict we accumulate into.
    # chunk : how many target IDs per API call. The target_chembl_id__in filter goes
    #         into the URL, which has a length limit, so we batch in groups of 50
    #         rather than sending hundreds at once.

    for i in range(0, len(ids), chunk):
        # step through ids in blocks: i = 0, 50, 100, ...
        batch = ids[i:i + chunk]
        # this block's slice (the last one may be shorter than 50 — slicing handles that).

        recs = target.filter(target_chembl_id__in=batch).only(
            ["target_chembl_id", "target_type", "organism", "target_components"])
  
        for t in recs:

            ttype = t.get("target_type")     # 'SINGLE PROTEIN' | 'PROTEIN COMPLEX' | ...
            org   = t.get("organism")         # e.g. 'Homo sapiens'

            accs = set() # uniprot accessions set

            keep = True
            if human_only and org != "Homo sapiens":# organism gate: when human_only, reject non-human targets 
                keep = False

            if single_protein_only and ttype != "SINGLE PROTEIN":# type gate: when single_protein_only (used for the noisy T_bio layer), reject complexes/families. 
                keep = False
            # T_mech passes single_protein_only=False, so
            # this gate is skipped and curated complexes are kept.

            if keep:
                accs = {c["accession"]
                        for c in (t.get("target_components") or [])
                        if c.get("accession")}
                # target_components is a LIST of protein parts of this target.
                #   (t.get(...) or [])  -> treat a missing/None component list as empty, so the loop is safe.
                #   if c.get("accession") -> skip components with no UniProt accession (rare non-protein components).
                # Result: the set of UniProt IDs for this one target. A single protein gives one; a complex/family gives several -> we union them per target.

            out[t["target_chembl_id"]] = {
                "target_type": ttype, "organism": org, "accessions": accs}
            # store under the target's ID. Keyed by target (not by accession) because
            # target_type is a property of the TARGET, not of an individual protein —
            # the same protein could appear in both a single-protein target and a complex.

    for tid in ids:
        out.setdefault(tid, {"target_type": None, "organism": None, "accessions": set()})
    # safety net: if the API didn't return a record for some requested ID (deleted/merged
    # target, or it simply wasn't in the response), give it an empty entry. This guarantees
    # every input ID has a key in `out`, so callers never hit a KeyError.

    return out

### Mechanism Targets - T_mech 

#### Step 1. molecule --> mechanisms rows
* Each surviving row from the mechanisms table hands a target_chembl_id.
#### Step 2. target ChEMBL ID --> UniProt accession(s). 
* Via the target table's target_components, to one or more UniProt accessions.

1. PARENT (parent ChEMBL id)  use mechanism.filter(parent_molecule_chembl_id=PARENT)
   ▼
2. mechanism rows ──filter: has target_chembl_id, molecular_mechanism, direct_interaction collect distinct target_chembl_id(s)

3. target.filter(target_chembl_id__in=[…])  →  per target: target_components → accession

4. T_mech : set[UniProt] union all accessions